# Synthesizer — curate the fan-out into a usable card

Takes the per-subject `PlannerOutput`s and produces a `SocialPrepResult`: per subject a
`framing`, `talking_points` (4-5 timeless), `news` (0-3 sourced), and a `reserve` pool.
One LLM call; selects/organizes, does not rewrite. `%run common.ipynb` for the bootstrap.
Developed against `synthesizer_fixture_sarah.json` so iteration is offline.

In [ ]:
import os
# Run from backend/ so `%run` and .env resolve regardless of the kernel's launch dir.
if not os.path.exists("common.ipynb"):
    for _p in ("backend", "extras/socialApp/backend"):
        if os.path.exists(f"{_p}/common.ipynb"):
            os.chdir(_p)
            break
%run common.ipynb


In [ ]:
# Synthesizer config + schemas (synthesizer-specific; not in common).
SYNTH_MODEL  = os.getenv("SYNTH_MODEL", "openai/gpt-4.1")
SYNTH_EFFORT = os.getenv("SYNTH_EFFORT", "none")
NEWS_MAX     = 3


class CuratedPoint(BaseModel):
    angle: str = Field(description="Copied VERBATIM from a candidate point. Do not rewrite.")
    context: str | None = Field(default=None, description="Copied verbatim from the candidate, or null.")
    source_url: str | None = Field(default=None, description="Copied verbatim from the candidate, or null.")

class SynthesizedSubject(BaseModel):
    subject: str
    framing: str = Field(description="Carried VERBATIM from the subject's framing.")
    talking_points: list[CuratedPoint] = Field(description="About 4-5 universally good points, each able to spark depth. Timeless, no source_url.")
    news: list[CuratedPoint] = Field(description="0-3 fresh, time-bound points, each WITH a source_url, that could be related or framed toward the person.")
    reserve: list[CuratedPoint] = Field(description="Remaining good candidates as a flat 'more like these' list for swapping. No ordering needed.")
    reasoning: str = Field(description="1-2 sentences: why THESE talking_points and news were chosen for this subject (selection rationale, not a subject summary or research notes).")

class SocialPrepResult(BaseModel):
    overall_context: str = Field(description="Carried VERBATIM. Shown first as the person thumbnail.")
    subjects: list[SynthesizedSubject]

print("CuratedPoint, SynthesizedSubject, SocialPrepResult defined.")

In [ ]:
# Cell 3 — Synthesizer agent + render + synthesize() + light guard

CARD_SYNTHESIZER_PROMPT = f"""\
You curate conversation-prep material into a usable card. A prior stage produced,
for each subject, a framing paragraph and many candidate talking points. Your job
is to SELECT and ORGANIZE the best of them — you do NOT research, and you do NOT
rewrite point text.

For reference, today's date is {TODAY}.

INPUT you receive:
  Overall context: <one thumbnail of the person the user is meeting>
  Then, per subject:
    Subject:  <name>
    Framing:  <intro paragraph>
    Candidate points, each with: angle / context / source_url

For each subject, produce three groups:

1) talking_points — about 4-5 universally good points.
   Pick points that almost anyone with this experience would have something to
   say about, and that open a door to a story, an opinion, or a memory rather
   than a yes/no answer. These are timeless: they carry NO source_url.

2) news — 0 to 3 fresh, time-bound points (the candidates that carry a
   source_url). Include news ONLY when it is genuinely recent and plausibly
   relevant to this person. Frame each news item TOWARD the person: something 
   that may have touched them and that they can react to ("did this affect you?"), 
   never a headline merely relayed. Every news point MUST keep its source_url.

3) reserve — a flat "more like these" list of the remaining good candidates, in
   case the user wants to swap one in. No ordering needed. Leave out only true
   duplicates and weak points.

4) reasoning — 1-2 sentences on WHY you chose these talking_points and news for
   this subject: what makes them the most universal / depth-sparking, and why the
   news is worth raising. This is your selection rationale — NOT a summary of the
   subject (that is the framing), and NOT research notes.

Light de-duplication: across all subjects, do not repeat the same question shape.
If the same underlying question shows up for several subjects, keep it for the
one subject it fits best and drop it from the others.

HARD RULES:
- Copy angle / context / source_url VERBATIM from the candidates. Never reword,
  never invent, never fabricate a URL.
- Carry each subject's framing and the overall_context VERBATIM.
- Every news point must have a source_url; talking_points must not.

WORKED EXAMPLE (one subject):

  Subject: Italy
  Framing: Italy is known for its food, ancient history, beautiful cities, and a
  relaxed, expressive lifestyle. Right now it is also dealing with widespread
  transport strikes and rising fuel and energy prices.

  talking_points:
    - angle: Food in Italy
      context: Italy is known for pizza, pasta, chocolates
    - angle: The Italian way of being
      context: Italians have a reputation for being expressive, social, and
      romantic.
    - angle: What landmark he visited
      context: Italy's famous attractions include the Colosseum in Rome, the Grand Canal in Venice, 
      and Florence's Duomo.
    - angle: What drew him to Italy in the first place
      context: None

  news:
    - angle: Transport strikes affecting the trip
      context: Italy is in the middle of widespread strikes - transport plus a
      recent general strike. Did it affect the trip?
      source_url: https://www.reuters.com/world/europe/italy-strikes/
    - angle: Whether Italy felt expensive right now, especially fuel and energy prices
      context: Italian inflation rose in May, driven largely by higher energy
      and transport costs.
      source_url: https://www.reuters.com/world/italy-eu-harmonised-cpi-may-2026/

  reserve:
    - angle: Regional differences he noticed between north and south
    - angle: Whether he picked up any Italian phrases

  reasoning: Food and culture work for almost anyone who took this trip and open
  into stories; "what landmark" and "what drew you here" turn toward his own
  experience. Both news items are in only because they plausibly touched his trip.

Why these were chosen: the four talking points work for almost anyone who took
this trip and each opens a door - food and culture give the user solid ground to 
stand on and an easy invitation for the other person to share their take, while 
"what landmark" and "what drew you here" turn toward his actual experience and 
motivation, which is where it gets personal and memorable.
The two news items are included only because they are current and likely to have
touched the trip directly: they let the user ask one specific, caring question
instead of generic small talk.

OUTPUT: fill SocialPrepResult (including each subject's reasoning). Keep subjects in the order given.
"""

card_synthesizer_agent = Agent(
    name="SynthesizerAgent",
    instructions=CARD_SYNTHESIZER_PROMPT,
    model=model(SYNTH_MODEL),
    model_settings=model_settings(SYNTH_EFFORT),
    output_type=SocialPrepResult,
)


def render_synth_input(overall_context: str, subjects: list[dict]) -> str:
    """subjects: [{subject, framing, points:[{angle, context, source_url}]}]"""
    lines = [f"Overall context: {overall_context}", ""]
    for s in subjects:
        lines.append(f"=== SUBJECT: {s['subject']} ===")
        lines.append(f"Framing: {s['framing']}")
        lines.append("Candidate points:")
        for pt in s["points"]:
            lines.append(f"- angle: {pt['angle']}")
            lines.append(f"  context: {pt.get('context') or '(none)'}")
            lines.append(f"  source_url: {pt.get('source_url') or '(none)'}")
        lines.append("")
    return "\n".join(lines)


async def synthesize(overall_context: str, subjects: list[dict]) -> SocialPrepResult:
    msg = render_synth_input(overall_context, subjects)
    r = await Runner.run(card_synthesizer_agent, msg, max_turns=2)
    return r.final_output


def guard(result: SocialPrepResult) -> None:
    for subj in result.subjects:
        for p in subj.news:
            if not p.source_url:
                print(f"[guard] {subj.subject}: news point without source_url: {p.angle[:60]}")
        for p in subj.talking_points:
            if p.source_url:
                print(f"[guard] {subj.subject}: talking_point carries a source_url: {p.angle[:60]}")
        if len(subj.news) > NEWS_MAX:
            print(f"[guard] {subj.subject}: {len(subj.news)} news > {NEWS_MAX}")
        selected_angles = [p.angle for p in subj.talking_points + subj.news]
        reserve_angles = {p.angle for p in subj.reserve}
        overlap = [a for a in selected_angles if a in reserve_angles]
        if overlap:
            print(f"[guard] {subj.subject}: {len(overlap)} point(s) in BOTH selected and reserve")
        if not (subj.reasoning or "").strip():
            print(f"[guard] {subj.subject}: empty reasoning")

print("card_synthesizer_agent + render + synthesize + guard defined.")

In [ ]:
# Demo / tests (standalone only) — gated so `%run` from the pipeline stays quiet.
RUN_DEMO = globals().get("RUN_DEMO", True)
if RUN_DEMO:
    # Cell 4 — Test 1: cross-subject dedup INCLUDING reserve (Finding 2), robustness over 3 runs
    import time

    fx = json.load(open("synthesizer_fixture_sarah.json"))
    subjects = [
        {"subject": s["subject"], "framing": s["planner"]["framing"],
         "points": [pt for th in s["planner"]["themes"] for pt in th["points"]]}
        for s in fx["subjects"]
    ]

    def is_ds_subject(name: str) -> bool:
        return "data scien" in name.lower()

    def bridge(angle: str) -> bool:
        """The 'connect this hobby to her data-science career' motif, any rewording."""
        a = angle.lower()
        return any(k in a for k in ("data scien", "data skill", "data-orient", "data orient", "data-driven"))

    def selected_points(subj):
        return subj.talking_points + subj.news

    def all_output_points(subj):
        return subj.talking_points + subj.news + subj.reserve

    def bridge_pts(res):
        return [(s.subject, p.angle) for s in res.subjects if not is_ds_subject(s.subject)
                for p in all_output_points(s) if bridge(p.angle)]

    in_pts = [(s["subject"], p["angle"]) for s in subjects if not is_ds_subject(s["subject"])
              for p in s["points"] if bridge(p["angle"])]
    print(f"INPUT career-bridge points across non-DS subjects: {len(in_pts)}")
    for subj, ang in in_pts:
        print(f"   [{subj}] {ang[:72]}")
    print()

    RUNS = 3
    runs = []
    for k in range(RUNS):
        t0 = time.monotonic()
        res = await synthesize(fx["overall_context"], subjects)
        el = time.monotonic() - t0
        runs.append(res)
        selected_b = [a for s in res.subjects if not is_ds_subject(s.subject)
                   for p in selected_points(s) if bridge(p.angle) for a in [p.angle]]
        all_b = bridge_pts(res)
        print(f"run {k+1}: {el:4.0f}s  career-bridge selected={len(selected_b)}  selected+reserve={len(all_b)}  "
              f"-> {'PASS' if len(all_b) <= 1 else 'FAIL'}")
        for subj, ang in all_b:
            print(f"     [{subj}] {ang[:72]}")

    result = runs[-1]  # keep last run for Cell 5 / Cell 6
    print()
    ok = all(len(bridge_pts(r)) <= 1 for r in runs)
    print("Test 1 (dedup incl. reserve, 3/3 runs) PASS" if ok else "Test 1 FAIL")

    print()
    for subj in result.subjects:
        print(f"--- {subj.subject}  (talking_points={len(subj.talking_points)} news={len(subj.news)} reserve={len(subj.reserve)}) ---")
        print("  talking_points:")
        for pt in subj.talking_points:
            print(f"    - {pt.angle}")
        print("  news:")
        for pt in subj.news:
            print(f"    - {pt.angle}  [src]")
    print()
    guard(result)


In [ ]:
# Demo / tests (standalone only) — gated so `%run` from the pipeline stays quiet.
RUN_DEMO = globals().get("RUN_DEMO", True)
if RUN_DEMO:
    # Cell 5 — Test 2: group constraints for universal topics + news
    # (reuses `result` and `subjects` from Cell 4 — no extra LLM call)
    input_by_subj = {s["subject"]: s["points"] for s in subjects}

    fails = 0
    for s in result.subjects:
        tp_no_src = all(p.source_url is None for p in s.talking_points)
        news_has_src = all(p.source_url for p in s.news)
        news_size_ok = len(s.news) <= NEWS_MAX
        tp_size_ok = 3 <= len(s.talking_points) <= 6
        ok = tp_no_src and news_has_src and news_size_ok and tp_size_ok
        fails += 0 if ok else 1
        print(f"  {s.subject:16} talking_points={len(s.talking_points)} news={len(s.news)} "
              f"tp_no_src={tp_no_src} news_has_src={news_has_src}"
              + ("" if ok else "  <-- FAIL"))
    print("Test 2 (universal topics + news constraints) PASS" if fails == 0 else f"Test 2 FAIL ({fails} subjects)")


In [ ]:
# Demo / tests (standalone only) — gated so `%run` from the pipeline stays quiet.
RUN_DEMO = globals().get("RUN_DEMO", True)
if RUN_DEMO:
    # Cell 6 — Test 3: reserve accounting + verbatim (no rewrite) + selected/reserve disjoint
    def norm(t):
        return (t or "").strip()

    fails = 0
    for s in result.subjects:
        inp = input_by_subj[s.subject]
        inp_angles = {norm(p["angle"]) for p in inp}
        selected_a = [norm(p.angle) for p in s.talking_points + s.news]
        res_a = [norm(p.angle) for p in s.reserve]
        out_a = selected_a + res_a
        disjoint = set(selected_a).isdisjoint(set(res_a))
        no_dup = len(out_a) == len(set(out_a))
        not_verbatim = [a for a in out_a if a not in inp_angles]
        dropped = len(inp) - len(out_a)
        ok = disjoint and no_dup and not not_verbatim and dropped >= 0
        fails += 0 if ok else 1
        print(f"  {s.subject:16} in={len(inp):2} selected={len(selected_a)} reserve={len(res_a)} "
              f"dropped={dropped} disjoint={disjoint} no_dup={no_dup} not_verbatim={len(not_verbatim)}"
              + ("" if ok else "  <-- FAIL"))
        for v in not_verbatim[:2]:
            print(f"       NOT-VERBATIM: {v[:80]}")
    print("Test 3 (accounting + verbatim) PASS" if fails == 0 else f"Test 3 FAIL ({fails} subjects)")
